# 09 — Ontology: Neighbourhood-Scoped Fact Tables
**Create filtered fact tables for the Bicycle_Fleet_Ontology graph**

The ontology's GraphQL layer cannot traverse relationships backed by large
fact tables (>200K rows). This notebook creates **neighbourhood-scoped copies**
of the 3 large fact tables plus the 2 summary tables, filtered to a single
neighbourhood. The dimension tables are already small enough.

### Strategy
| Original Table | Rows | Ontology Table | Expected Rows |
|---|---|---|---|
| `fact_availability` | ~560M | `onto_fact_availability` | ~15-20M |
| `fact_hourly_demand` | ~200K | `onto_fact_hourly_demand` | ~7K |
| `fact_rebalancing` | ~500K | `onto_fact_rebalancing` | ~17K |
| `fact_weather_impact` | ~50K | (keep original) | ~50K |
| `forecast_demand` | ~2K | (keep original) | ~2K |
| `gold_station_snapshot` | ~120 | `onto_station_snapshot` | ~4-8 |
| `gold_availability_recent` | ~2.8K | `onto_availability_recent` | ~100-200 |
| `dim_station` | ~120 | `onto_dim_station` | ~4-8 |

After running, rebind the ontology entity types to the `onto_*` tables in the UI.

### Prerequisites
1. Attach **bicycles_gold** as the default lakehouse
2. Gold Star Schema (NB04) and ML Forecast (NB06) must have run

In [ ]:
# ============================================================
# CELL 1 — Configuration & Neighbourhood Selection
# ============================================================

from pyspark.sql.functions import col, count, lit
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

GOLD = "bicycles_gold"

# ── Attach default lakehouse context (required when running via pipeline) ──
spark.sql(f"USE {GOLD}")
print(f"Default catalog/database set to: {GOLD}")

# ── Show all neighbourhoods with station counts ──
print("=== NEIGHBOURHOODS ===")
df_nbhd = spark.sql(f"""
    SELECT 
        n.neighbourhood_key,
        n.neighbourhood_name,
        n.station_count,
        n.total_capacity,
        n.health_score,
        n.density_tier
    FROM {GOLD}.dim_neighbourhood n
    ORDER BY n.station_count DESC
""")
df_nbhd.show(50, truncate=False)

# ── Show fact table row counts per neighbourhood ──
print("\n=== FACT_AVAILABILITY ROWS PER NEIGHBOURHOOD (sampled) ===")
spark.sql(f"""
    SELECT ds.neighbourhood, COUNT(*) as availability_rows
    FROM {GOLD}.fact_availability fa
    JOIN {GOLD}.dim_station ds ON fa.station_key = ds.station_key
    GROUP BY ds.neighbourhood
    ORDER BY availability_rows DESC
""").show(50, truncate=False)

print("\n=== FACT_REBALANCING ROWS PER NEIGHBOURHOOD ===")
spark.sql(f"""
    SELECT neighbourhood, COUNT(*) as rebal_rows
    FROM {GOLD}.fact_rebalancing
    GROUP BY neighbourhood
    ORDER BY rebal_rows DESC
""").show(50, truncate=False)

print("\n=== FACT_HOURLY_DEMAND ROWS PER NEIGHBOURHOOD ===")
spark.sql(f"""
    SELECT n.neighbourhood_name, COUNT(*) as demand_rows
    FROM {GOLD}.fact_hourly_demand d
    JOIN {GOLD}.dim_neighbourhood n ON d.neighbourhood_key = n.neighbourhood_key
    GROUP BY n.neighbourhood_name
    ORDER BY demand_rows DESC
""").show(50, truncate=False)

In [ ]:
# ============================================================
# CELL 2 — Pick the neighbourhood
# ============================================================
# 
# After reviewing Cell 1 output, set your target neighbourhood here.
# Good choices for a demo:
#   - A neighbourhood with 4-8 stations (not too few, not too many)
#   - Recognizable name (Hyde Park, Kensington, Shoreditch, etc.)
#   - Good variety of data (demand spikes, rebalancing, weather impact)
#
# ⚡ CHANGE THIS to your chosen neighbourhood ⚡

FOCUS_NEIGHBOURHOOD = "Sands End"   # ← Change this after reviewing Cell 1

print(f"Target neighbourhood: {FOCUS_NEIGHBOURHOOD}")

# Validate it exists
nbhd_row = spark.sql(f"""
    SELECT neighbourhood_key, neighbourhood_name, station_count, total_capacity
    FROM {GOLD}.dim_neighbourhood
    WHERE neighbourhood_name = '{FOCUS_NEIGHBOURHOOD}'
""")
nbhd_row.show(truncate=False)
assert nbhd_row.count() > 0, f"Neighbourhood '{FOCUS_NEIGHBOURHOOD}' not found!"

FOCUS_NBHD_KEY = nbhd_row.first()["neighbourhood_key"]
print(f"neighbourhood_key = {FOCUS_NBHD_KEY}")

# Get the station keys for this neighbourhood
station_keys = [row["station_key"] for row in spark.sql(f"""
    SELECT station_key FROM {GOLD}.dim_station
    WHERE neighbourhood = '{FOCUS_NEIGHBOURHOOD}'
""").collect()]
print(f"Station keys: {station_keys} ({len(station_keys)} stations)")

In [ ]:
# ============================================================
# CELL 3 — Create filtered dimension: onto_dim_station
# ============================================================
# Only stations in the focus neighbourhood

print("Creating onto_dim_station...")
df_station = spark.sql(f"""
    SELECT * FROM {GOLD}.dim_station
    WHERE neighbourhood = '{FOCUS_NEIGHBOURHOOD}'
""")
df_station.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD}.onto_dim_station")
cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {GOLD}.onto_dim_station").first()["cnt"]
print(f"  onto_dim_station: {cnt} rows ✓")

In [ ]:
# ============================================================
# CELL 4 — Create filtered fact: onto_fact_availability
# ============================================================
# Only availability events for stations in the focus neighbourhood.
# This is the BIG one — 560M rows down to ~15-20M.
# We also limit to last 7 days to keep it truly ontology-friendly.

from pyspark.sql.functions import current_timestamp, expr

print("Creating onto_fact_availability (last 7 days, focus neighbourhood only)...")
df_avail = spark.sql(f"""
    SELECT fa.*
    FROM {GOLD}.fact_availability fa
    WHERE fa.station_key IN ({','.join(str(k) for k in station_keys)})
      AND fa.event_timestamp >= current_timestamp() - INTERVAL 7 DAYS
""")
df_avail.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD}.onto_fact_availability")
cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {GOLD}.onto_fact_availability").first()["cnt"]
print(f"  onto_fact_availability: {cnt} rows ✓")
print(f"  (Goal: < 50,000 rows for GraphQL)")

In [ ]:
# ============================================================
# CELL 5 — Create filtered fact: onto_fact_hourly_demand
# ============================================================

print("Creating onto_fact_hourly_demand...")
df_demand = spark.sql(f"""
    SELECT d.*
    FROM {GOLD}.fact_hourly_demand d
    WHERE d.neighbourhood_key = {FOCUS_NBHD_KEY}
""")
df_demand.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD}.onto_fact_hourly_demand")
cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {GOLD}.onto_fact_hourly_demand").first()["cnt"]
print(f"  onto_fact_hourly_demand: {cnt} rows ✓")

In [ ]:
# ============================================================
# CELL 6 — Create filtered fact: onto_fact_rebalancing  
# ============================================================

print("Creating onto_fact_rebalancing...")
df_rebal = spark.sql(f"""
    SELECT r.*
    FROM {GOLD}.fact_rebalancing r
    WHERE r.station_key IN ({','.join(str(k) for k in station_keys)})
""")
df_rebal.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD}.onto_fact_rebalancing")
cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {GOLD}.onto_fact_rebalancing").first()["cnt"]
print(f"  onto_fact_rebalancing: {cnt} rows ✓")

In [ ]:
# ============================================================
# CELL 7 — Create filtered summary: onto_station_snapshot
# ============================================================
# After copying, seed 1 station as CRITICAL (is_critical=true, bikes=0)
# and 1 station with surplus bikes (>10) so Activator rules can trigger.

print("Creating onto_station_snapshot...")
df_snap = spark.sql(f"""
    SELECT * FROM {GOLD}.gold_station_snapshot
    WHERE neighbourhood = '{FOCUS_NEIGHBOURHOOD}'
""")
df_snap.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD}.onto_station_snapshot")

cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {GOLD}.onto_station_snapshot").first()["cnt"]
print(f"  onto_station_snapshot: {cnt} rows ✓")

# ── Demo-seed: force 1 station CRITICAL for Activator rule ──
print("  Seeding Activator trigger data...")

_first_key = spark.sql(f"SELECT MIN(station_key) AS k FROM {GOLD}.onto_station_snapshot").first()["k"]
_last_key  = spark.sql(f"SELECT MAX(station_key) AS k FROM {GOLD}.onto_station_snapshot").first()["k"]

# Station 1 → CRITICAL (empty station)
spark.sql(f"""
    UPDATE {GOLD}.onto_station_snapshot
    SET is_critical = true,
        bikes_available = 0,
        utilization_pct = 0.0,
        availability_band = 'empty',
        operational_status = 'EMPTY',
        needs_attention = true,
        last_event_type = 'empty_station'
    WHERE station_key = {_first_key}
""")
print(f"    ✓ station_key={_first_key} → is_critical=true, bikes=0")

# Station 2 → SURPLUS (bikes > 10 for Fleet Surplus rule)
spark.sql(f"""
    UPDATE {GOLD}.onto_station_snapshot
    SET bikes_available = 15,
        is_critical = false,
        operational_status = 'NORMAL',
        availability_band = 'balanced',
        utilization_pct = 0.55
    WHERE station_key = {_last_key}
""")
print(f"    ✓ station_key={_last_key} → bikes_available=15 (fleet surplus)")

In [ ]:
# ============================================================
# CELL 8 — Create filtered summary: onto_availability_recent
# ============================================================

print("Creating onto_availability_recent...")
df_recent = spark.sql(f"""
    SELECT * FROM {GOLD}.gold_availability_recent
    WHERE neighbourhood = '{FOCUS_NEIGHBOURHOOD}'
""")
df_recent.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD}.onto_availability_recent")
cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {GOLD}.onto_availability_recent").first()["cnt"]
print(f"  onto_availability_recent: {cnt} rows ✓")

In [ ]:
# ============================================================
# CELL 9 — Create filtered: onto_fact_weather_impact
# ============================================================

print("Creating onto_fact_weather_impact...")
df_wi = spark.sql(f"""
    SELECT wi.*
    FROM {GOLD}.fact_weather_impact wi
    WHERE wi.neighbourhood_key = {FOCUS_NBHD_KEY}
""")
df_wi.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD}.onto_fact_weather_impact")
cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {GOLD}.onto_fact_weather_impact").first()["cnt"]
print(f"  onto_fact_weather_impact: {cnt} rows ✓")

In [ ]:
# ============================================================
# CELL 10 — Create filtered: onto_forecast_demand
# ============================================================
# After copying, seed rush-hour rows with pre_position_recommended=true
# so the "High Demand Forecast" Activator rule can trigger.

print("Creating onto_forecast_demand...")
df_fc = spark.sql(f"""
    SELECT f.*
    FROM {GOLD}.forecast_demand f
    WHERE f.neighbourhood = '{FOCUS_NEIGHBOURHOOD}'
""")
df_fc.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD}.onto_forecast_demand")

cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {GOLD}.onto_forecast_demand").first()["cnt"]
print(f"  onto_forecast_demand: {cnt} rows ✓")

# ── Demo-seed: flag rush-hour forecasts for Activator ──
print("  Seeding pre_position_recommended for rush hours...")
spark.sql(f"""
    UPDATE {GOLD}.onto_forecast_demand
    SET pre_position_recommended = true,
        demand_tier = 'High'
    WHERE hour_of_day IN (8, 9, 17, 18)
""")
_prepos = spark.sql(f"SELECT COUNT(*) AS c FROM {GOLD}.onto_forecast_demand WHERE pre_position_recommended = true").first()["c"]
print(f"    ✓ {_prepos} rows now have pre_position_recommended=true")

In [ ]:
# ============================================================
# CELL 11 — Refresh Semantic Model (Direct Lake)
# ============================================================
# Trigger a refresh of the "Bicycle Ontology Model" so Activator
# picks up the newly written onto_* tables via Direct Lake.

import requests as _req
import sempy.fabric as _fabric

# Dynamic lookup — no hardcoded IDs
_WS_ID = _fabric.get_workspace_id()
_SM_NAME = "Bicycle Ontology Model"
_SM_ID = None
try:
    _token = notebookutils.credentials.getToken("pbi")
    _headers = {"Authorization": f"Bearer {_token}", "Content-Type": "application/json"}
    _items_resp = _req.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{_WS_ID}/semanticModels",
        headers=_headers,
    )
    if _items_resp.status_code == 200:
        for _item in _items_resp.json().get("value", []):
            if _item["displayName"] == _SM_NAME:
                _SM_ID = _item["id"]
                break
    if _SM_ID:
        _url = f"https://api.powerbi.com/v1.0/myorg/groups/{_WS_ID}/datasets/{_SM_ID}/refreshes"
        _resp = _req.post(_url, headers=_headers, json={"notifyOption": "NoNotification"})
        if _resp.status_code == 202:
            print("  \u2713 Semantic model refresh triggered (202 Accepted)")
        else:
            print(f"  [WARN] SM refresh returned {_resp.status_code}: {_resp.text[:200]}")
    else:
        print(f"  [WARN] Semantic Model '{_SM_NAME}' not found in workspace")
        print("  \u2192 Refresh manually: Fabric UI \u2192 Bicycle Ontology Model \u2192 Refresh now")
except Exception as e:
    print(f"  [WARN] Could not refresh semantic model: {e}")
    print("  \u2192 Refresh manually: Fabric UI \u2192 Bicycle Ontology Model \u2192 Refresh now")

In [ ]:
# ============================================================
# CELL 12 — Summary & Rebinding Instructions
# ============================================================

print("=" * 60)
print(f"  ONTOLOGY TABLES CREATED — {FOCUS_NEIGHBOURHOOD}")
print("=" * 60)

tables = [
    "onto_dim_station",
    "onto_fact_availability",
    "onto_fact_hourly_demand",
    "onto_fact_rebalancing",
    "onto_station_snapshot",
    "onto_availability_recent",
    "onto_fact_weather_impact",
    "onto_forecast_demand",
]

print(f"\n{'Table':<35} {'Rows':>10}")
print("-" * 47)
for t in tables:
    cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {GOLD}.{t}").first()["cnt"]
    status = "✓" if cnt < 50000 else "⚠️ >50K"
    print(f"  {t:<33} {cnt:>10,}  {status}")

print(f"\nDimension tables (unchanged — already small):")
for dim in ["dim_neighbourhood", "dim_time", "dim_date", "dim_weather"]:
    cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {GOLD}.{dim}").first()["cnt"]
    print(f"  {dim:<33} {cnt:>10,}  ✓")

print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  NEXT STEPS: Rebind Ontology Entity Types
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In the Fabric UI, open Bicycle_Fleet_Ontology and update
each entity type's data binding to point to the onto_* tables:

  Entity Type             Old Table                → New Table
  ─────────────────────   ───────────────────────   ──────────────────────
  Station                 dim_station              → onto_dim_station
  AvailabilityEvent       fact_availability        → onto_fact_availability
  HourlyDemand            fact_hourly_demand       → onto_fact_hourly_demand
  RebalancingAssessment   fact_rebalancing         → onto_fact_rebalancing
  StationSnapshot         gold_station_snapshot    → onto_station_snapshot
  RecentAvailability      gold_availability_recent → onto_availability_recent
  WeatherImpact           fact_weather_impact      → onto_fact_weather_impact
  DemandForecast          forecast_demand          → onto_forecast_demand

  KEEP UNCHANGED (already small):
  Neighbourhood           dim_neighbourhood        (unchanged)
  TimeSlot                dim_time                 (unchanged)
  CalendarDate            dim_date                 (unchanged)
  WeatherObservation      dim_weather              (unchanged)

Also update the Graph Model data sources to point to the onto_* tables.
""")